In [1]:
import os
import math
import numpy as np
import torch
import torch.nn as nn
from torch.nn import init
import torch.nn.functional as F
from einops import rearrange
import cv2
from tqdm.notebook import tqdm
from typing import List
import matplotlib.pyplot as plt
import glob
import json
import gc

In [2]:
class Identity_(nn.Module):
    def forward(self, inputs): return inputs

def _act(act_type, **kwargs):
    if act_type is None or act_type == "Identity": return Identity_()
    elif act_type == "ReLU": return nn.ReLU(inplace=True)
    raise Exception("Not implemented !!!")

class Linear_(nn.Module):
    def __init__(self, in_features, out_features, bias=True, act="ReLU", is_folded=True):
        super(Linear_, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.bias = bias
        self.act_type = act
        self.is_folded = is_folded
        self.linear = nn.Linear(in_features=self.in_features, out_features=self.out_features, bias=self.bias)
        self.act = _act(self.act_type)
    def forward(self, inputs): return self.act(self.linear(inputs))

def gather_nd(params, indices):
    orig_shape = list(indices.shape)
    num_samples = np.prod(orig_shape[:-1])
    m = orig_shape[-1]
    n = len(params.shape)
    if m <= n: out_shape = orig_shape[:-1] + list(params.shape)[m:]
    else: raise ValueError(f'm > n')
    indices = indices.reshape((num_samples, m)).transpose(0, 1).tolist()
    output = params[tuple(indices)] # Đã fix cảnh báo tuple của PyTorch
    return output.reshape(out_shape).contiguous()

class FrameSimilarity(nn.Module):
    def __init__(self, in_channels, inner_channels, similarity_dim=128, lookup_window=101, output_dim=128, stop_gradient=False, use_bias=True):
        super(FrameSimilarity, self).__init__()
        self.projection = Linear_(in_features=in_channels, out_features=similarity_dim, bias=use_bias, act="Identity")
        self.fc = Linear_(in_features=inner_channels, out_features=output_dim, bias=True, act="ReLU")
        self.lookup_window = lookup_window
        self.stop_gradient = stop_gradient
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def forward(self, inputs):
        x = torch.cat([torch.mean(x, dim=[3, 4]) for x in inputs], dim=1)
        if self.stop_gradient: x = x.detach()
        x = x.permute(dims=[0, 2, 1])
        batch_size, time_window, old_channels = x.shape
        x = x.reshape(shape=[batch_size * time_window, old_channels])
        x = self.projection(x)
        x = F.normalize(x, p=2, dim=1)
        _, new_channels = x.shape
        x = x.reshape(shape=[batch_size, time_window, new_channels])
        y = x.permute(dims=[0, 2, 1])
        similarities = torch.matmul(x, y)
        similarities_padded = F.pad(similarities, pad=[(self.lookup_window - 1) // 2, (self.lookup_window - 1) // 2, 0, 0, 0, 0])
        batch_indices = torch.arange(0, batch_size, device=self.device).reshape(shape=[batch_size, 1, 1]).repeat([1, time_window, self.lookup_window])
        time_indices = torch.arange(0, time_window, device=self.device).reshape(shape=[1, time_window, 1]).repeat([batch_size, 1, self.lookup_window])
        lookup_indices = torch.arange(0, self.lookup_window, device=self.device).reshape(shape=[1, 1, self.lookup_window]).repeat([batch_size, time_window, 1]) + time_indices
        indices = torch.stack([batch_indices, time_indices, lookup_indices], dim=-1)
        return self.fc(gather_nd(similarities_padded, indices))

class ColorHistograms(nn.Module):
    def __init__(self, lookup_window=101, output_dim=128):
        super(ColorHistograms, self).__init__()
        self.fc = Linear_(in_features=101, out_features=output_dim, bias=True, act="ReLU") if output_dim is not None else None
        self.lookup_window = lookup_window
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def unsorted_segment_sum(self, data, segment_ids, num_segments):
        if len(segment_ids.shape) == 1:
            s = torch.prod(torch.tensor(data.shape[1:])).long()
            segment_ids = segment_ids.repeat_interleave(s).view(segment_ids.shape[0], *data.shape[1:])
        shape = [num_segments] + list(data.shape[1:])
        return torch.zeros(*shape, device=self.device).scatter_add(0, segment_ids, data.float()).type(data.dtype)

    def compute_color_histograms(self, frames):
        frames = frames.type(dtype=torch.int32).permute(0, 2, 3, 4, 1)
        def get_bin(frames):
            R, G, B = frames[:, :, 0] >> 5, frames[:, :, 1] >> 5, frames[:, :, 2] >> 5
            return (R << 6) + (G << 3) + B
        batch_size, time_window, height, width = frames.shape[0], frames.shape[1], frames.shape[2], frames.shape[3]
        frames_flatten = frames.reshape(shape=[batch_size * time_window, height * width * (2 if frames.shape[-1] == 6 else 1), 3])
        binned_values = get_bin(frames_flatten) + (torch.arange(0, batch_size * time_window, dtype=torch.int32, device=self.device) << 9).unsqueeze(dim=-1)
        ones = torch.ones_like(binned_values, dtype=torch.int32, device=self.device)
        histograms = self.unsorted_segment_sum(ones, segment_ids=binned_values.type(dtype=torch.long), num_segments=batch_size * time_window * 512)
        histograms_normalized = torch.sum(histograms, dim=1).reshape(shape=[batch_size, time_window, 512]).type(dtype=torch.float32)
        return histograms_normalized / torch.norm(histograms_normalized, dim=2, keepdim=True)

    def forward(self, inputs):
        x = self.compute_color_histograms(inputs)
        batch_size, time_window = x.shape[0], x.shape[1]
        similarities_padded = F.pad(torch.matmul(x, x.permute(dims=[0, 2, 1])), pad=[(self.lookup_window - 1) // 2, (self.lookup_window - 1) // 2, 0, 0, 0, 0])
        batch_indices = torch.arange(0, batch_size, device=self.device).reshape(shape=[batch_size, 1, 1]).repeat([1, time_window, self.lookup_window])
        time_indices = torch.arange(0, time_window, device=self.device).reshape(shape=[1, time_window, 1]).repeat([batch_size, 1, self.lookup_window])
        lookup_indices = torch.arange(0, self.lookup_window, device=self.device).reshape(shape=[1, 1, self.lookup_window]).repeat([batch_size, time_window, 1]) + time_indices
        similarities = gather_nd(similarities_padded, torch.stack([batch_indices, time_indices, lookup_indices], dim=-1))
        return self.fc(similarities) if self.fc is not None else similarities

class Conv3DConfigurable(nn.Module):
    def __init__(self, in_channels, filters, dilation_rate, mid_filter=None, separable=True, sharable=False, octave=False, use_bias=False):
        super(Conv3DConfigurable, self).__init__()
        self.layers = nn.ModuleList()
        if separable:
            if not sharable:
                conv1 = torch.nn.Conv3d(in_channels=in_channels, out_channels=2 * filters if mid_filter is None else mid_filter, kernel_size=(1, 3, 3), padding=(0, 1, 1), dilation=(1, 1, 1), bias=False)
                init.kaiming_normal_(conv1.weight, mode="fan_in", nonlinearity="relu")
                self.layers.append(conv1)
            conv2 = torch.nn.Conv3d(in_channels=2 * filters if mid_filter is None else mid_filter, out_channels=filters, kernel_size=(3, 1, 1), padding=(1 * dilation_rate, 0, 0), dilation=(dilation_rate, 1, 1), bias=use_bias)
            init.kaiming_normal_(conv2.weight, mode="fan_in", nonlinearity="relu")
            self.layers.append(conv2)

    def forward(self, inputs):
        x = inputs
        for layer in self.layers: x = layer(x)
        return x

class DilatedDCNNV2ABC(nn.Module):
    def __init__(self, in_channels, filters, batch_norm=True, activation=F.relu, octave_conv=False, multiplier=4, n_dilation=4, st_type="A"):
        super(DilatedDCNNV2ABC, self).__init__()
        self.share = torch.nn.Conv3d(in_channels=in_channels, out_channels=multiplier * filters, kernel_size=(1, 3, 3), padding=(0, 1, 1), dilation=(1, 1, 1), bias=False)
        init.kaiming_normal_(self.share.weight, mode="fan_in", nonlinearity="relu")
        self.conv_blocks = nn.ModuleList()
        n_in_plane = multiplier * filters if st_type != "B" else in_channels
        n_filter_per_module = (filters * 4) // n_dilation
        for dilation in range(n_dilation-1):
            self.conv_blocks.append(Conv3DConfigurable(n_in_plane, n_filter_per_module, 2 ** dilation, mid_filter=n_in_plane, separable=True, sharable=True, use_bias=not batch_norm, octave=octave_conv))
        self.conv_blocks.append(Conv3DConfigurable(n_in_plane, (filters * 4) - n_filter_per_module * (n_dilation-1), 2 ** (n_dilation - 1), mid_filter=n_in_plane, separable=True, sharable=True, use_bias=not batch_norm, octave=octave_conv))
        self.st_type = st_type
        self.batch_norm = torch.nn.BatchNorm3d(num_features=filters * 4, eps=1e-3, momentum=0.1) if batch_norm else None
        self.activation = activation

    def forward(self, inputs):
        feature = self.share(inputs)
        if self.st_type == "A": x = torch.cat([block(feature) for block in self.conv_blocks], dim=1)
        elif self.st_type == "B": x = torch.cat([block(inputs) for block in self.conv_blocks], dim=1) + feature
        elif self.st_type == "C": x = torch.cat([block(feature) for block in self.conv_blocks], dim=1) + feature
        if self.batch_norm is not None: x = self.batch_norm(x)
        if self.activation is not None: x = self.activation(x)
        return x

class DilatedDCNNV2(nn.Module):
    def __init__(self, in_channels, filters, multiplier=2, n_dilation=4, batch_norm=True, activation=F.relu, octave_conv=False):
        super(DilatedDCNNV2, self).__init__()
        self.conv_blocks = nn.ModuleList()
        n_filter_per_module = (filters * 4) // n_dilation
        for dilation in range(n_dilation-1):
            self.conv_blocks.append(Conv3DConfigurable(in_channels, n_filter_per_module, mid_filter=multiplier*filters, dilation_rate=2 ** dilation, use_bias=not batch_norm, octave=octave_conv))
        self.conv_blocks.append(Conv3DConfigurable(in_channels, (filters * 4) - n_filter_per_module * (n_dilation-1), mid_filter=multiplier*filters, dilation_rate=2 ** (n_dilation-1), use_bias=not batch_norm, octave=octave_conv))
        self.batch_norm = torch.nn.BatchNorm3d(num_features=filters * 4, eps=1e-3, momentum=0.1) if batch_norm else None
        self.activation = activation

    def forward(self, inputs):
        x = torch.cat([block(inputs) for block in self.conv_blocks], dim=1)
        if self.batch_norm is not None: x = self.batch_norm(x)
        if self.activation is not None: x = self.activation(x)
        return x

class Attention1D(nn.Module):
    def __init__(self, dim_in, dim_out, num_heads, qkv_bias=False, attn_drop=0., proj_drop=0., with_cls_token=False, skip_conv_proj=False, n_layer=1, **kwargs):
        super().__init__()
        self.num_heads = num_heads
        self.n_layer = n_layer
        self.scale = dim_out ** -0.5
        self.proj_q, self.proj_k, self.proj_v, self.attn_drop, self.proj, self.proj_drop = nn.ModuleList(), nn.ModuleList(), nn.ModuleList(), nn.ModuleList(), nn.ModuleList(), nn.ModuleList()
        for _ in range(n_layer):
            self.proj_q.append(nn.Linear(dim_in, dim_out, bias=qkv_bias))
            self.proj_k.append(nn.Linear(dim_in, dim_out, bias=qkv_bias))
            self.proj_v.append(nn.Linear(dim_in, dim_out, bias=qkv_bias))
            self.attn_drop.append(nn.Dropout(attn_drop))
            self.proj.append(nn.Linear(dim_out, dim_out))
            self.proj_drop.append(nn.Dropout(proj_drop))
            dim_in = dim_out

    def forward(self, x, t, h, w):
        x = rearrange(x, 'b c t H W -> b t (c H W)')
        if self.n_layer == 0: return None
        for idx in range(self.n_layer):
            q = rearrange(self.proj_q[idx](x), 'b t (h d) -> b h t d', h=self.num_heads)
            k = rearrange(self.proj_k[idx](x), 'b t (h d) -> b h t d', h=self.num_heads)
            v = rearrange(self.proj_v[idx](x), 'b t (h d) -> b h t d', h=self.num_heads)
            attn_score = torch.einsum('bhlk,bhtk->bhlt', [q, k]) * self.scale
            attn = self.attn_drop[idx](F.softmax(attn_score, dim=-1))
            x = self.proj_drop[idx](self.proj[idx](rearrange(torch.einsum('bhlt,bhtv->bhlv', [attn, v]), 'b h t d -> b t (h d)')))
        return x

class TransNetV2Supernet(nn.Module):
    def __init__(self, D=1024, use_many_hot_targets=True, use_frame_similarity=True, use_mean_pooling=None, dropout_rate=0.5, use_resnet_like_top=False, frame_similarity_on_last_layer=False, use_color_histograms=True, chromosome=None):
        super(TransNetV2Supernet, self).__init__()
        self.reprocess_layer = (lambda x: x / 255.)
        self.Layer_0_3 = DilatedDCNNV2(3, 16, multiplier=1)
        self.Layer_1_8 = DilatedDCNNV2ABC(16 * 4, 16, multiplier=4, n_dilation=5, st_type="A")
        self.Layer_2_8 = DilatedDCNNV2ABC(16 * 4, 32, multiplier=4, n_dilation=5, st_type="A")
        self.Layer_3_8 = DilatedDCNNV2ABC(32 * 4, 32, multiplier=4, n_dilation=5, st_type="A")
        self.Layer_4_13 = DilatedDCNNV2(32 * 4, 64, multiplier=3, n_dilation=5)
        self.Layer_5_12 = DilatedDCNNV2(64 * 4, 64, multiplier=2, n_dilation=5)
        self.Layer_6_0 = Attention1D(dim_in=256 * 3 * 6, dim_out=1024, num_heads=4, qkv_bias=False, attn_drop=0., proj_drop=0., with_cls_token=False, skip_conv_proj=True, n_layer=0)
        self.pool = torch.nn.AvgPool3d(kernel_size=(1, 2, 2))
        
        in_features = 4864 if use_frame_similarity and use_color_histograms else 4736 if use_frame_similarity or use_color_histograms else 4608
        self.fc1_0 = Linear_(in_features=in_features, out_features=D, bias=True, act="ReLU")
        self.fc1 = Linear_(in_features=in_features + 1024, out_features=D, bias=True, act="ReLU")
        self.cls_layer1 = Linear_(in_features=1024, out_features=1, bias=True, act="Identity")
        self.cls_layer2 = Linear_(in_features=1024, out_features=1, bias=True, act="Identity") if use_many_hot_targets else None
        
        self.frame_sim_layer = FrameSimilarity(in_channels=448, inner_channels=101) if use_frame_similarity else None
        self.color_hist_layer = ColorHistograms(lookup_window=101, output_dim=128) if use_color_histograms else None
        self.use_mean_pooling = use_mean_pooling
        self.dropout = torch.nn.Dropout(p=1.0 - dropout_rate) if dropout_rate is not None else None
        self.frame_similarity_on_last_layer = frame_similarity_on_last_layer

    def forward(self, inputs, **kwargs):
        x = self.reprocess_layer(inputs)
        block_features = []
        shortcut = None
        for layer_index, op in enumerate([self.Layer_0_3, self.Layer_1_8, self.Layer_2_8, self.Layer_3_8, self.Layer_4_13, self.Layer_5_12]):
            x = op(x)
            if layer_index in [0, 2, 4]: shortcut = x
            else:
                x = shortcut + x
                x = self.pool(x)
                block_features.append(x)
        transf_x = self.Layer_6_0(x, t=x.size(1), h=3, w=6)

        # Đã tách lệnh để fix lỗi chiều tensor
        if self.use_mean_pooling: 
            x = torch.mean(x, dim=[3, 4])
        else: 
            x = x.permute(dims=[0, 2, 3, 4, 1])
            x = x.reshape(shape=[x.shape[0], x.shape[1], np.prod(list(x.shape)[2:])])

        if self.frame_sim_layer and not self.frame_similarity_on_last_layer:
            x = torch.cat([self.frame_sim_layer(block_features), x], dim=2)
        if self.color_hist_layer:
            x = torch.cat([self.color_hist_layer(inputs), x], dim=2)
        
        if transf_x is not None: x = self.fc1(torch.cat([transf_x, x], dim=2))
        else: x = self.fc1_0(x)

        if self.dropout: x = self.dropout(x)
        if self.frame_sim_layer and self.frame_similarity_on_last_layer:
            x = torch.cat([self.frame_sim_layer(block_features), x], dim=2)

        one_hot = self.cls_layer1(x)
        if self.cls_layer2 is not None: return one_hot, self.cls_layer2(x)
        return one_hot


def get_frames(video_file_path: str, width: int = 48, height: int = 27) -> np.ndarray:
    """Đọc video và chuyển thành ma trận numpy chống lỗi cv2"""
    cap = cv2.VideoCapture(video_file_path)
    frames = []
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        resized = cv2.resize(frame, (width, height))
        frames.append(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
    cap.release()
    return np.array(frames, dtype=np.uint8)

In [3]:
def get_batches(frames: np.ndarray):
    """Cắt thành batch nhỏ (100 frame) để chống tràn RAM Kaggle"""
    reminder = 50 - len(frames) % 50
    if reminder == 50: reminder = 0
    frames = np.concatenate([frames[:1]] * 25 + [frames] + [frames[-1:]] * (reminder + 25), 0)
    for i in range(0, len(frames) - 50, 50):
        yield frames[i:i + 100]

class AutoShot:
    def __init__(self, pretrained_path: str):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model = TransNetV2Supernet().eval()
        if os.path.exists(pretrained_path):
            model_dict = self.model.state_dict()
            pretrained_dict = torch.load(pretrained_path, map_location=self.device, weights_only=True)
            pretrained_dict = {k: v for k, v in pretrained_dict.get('net', pretrained_dict).items() if k in model_dict}
            model_dict.update(pretrained_dict)
            self.model.load_state_dict(model_dict)
        self.model.to(self.device)

    def detect_shots(self, frames: np.ndarray) -> List[List[int]]:
        """Nhận diện điểm chuyển cảnh và trả về danh sách thô (Raw Output)"""
        predictions = []
        for batch in tqdm(get_batches(frames), desc="Chạy Model AutoShot", unit="batch"):
            with torch.no_grad():
                batch_tensor = torch.from_numpy(batch.transpose((3, 0, 1, 2))[np.newaxis, ...]) * 1.0
                one_hot = self.model(batch_tensor.to(self.device))
                if isinstance(one_hot, tuple): one_hot = one_hot[0]
                predictions.append(torch.sigmoid(one_hot[0]).cpu().numpy()[25:75])
        
        predictions = np.concatenate(predictions, axis=0)[:len(frames)]
        preds_binary = (predictions > 0.5).astype(np.uint8)
        
        scenes, t_prev, start = [], 0, 0
        for i, t in enumerate(preds_binary):
            if t_prev == 1 and t == 0: start = i
            if t_prev == 0 and t == 1 and i != 0: scenes.append([start, i])
            t_prev = t
        scenes.append([start, len(predictions) - 1])
        return scenes if scenes else [[0, len(predictions) - 1]]

In [4]:
# !ls /kaggle/input/datasets/

In [5]:
# Đường dẫn gốc chứa toàn bộ các thư mục batch
PARENT_VIDEO_DIR = "/kaggle/input/datasets/dtdat1725/aic-hcmc-2025-batch-1/video"
WEIGHTS_PATH = "/kaggle/input/datasets/anhnguynnhtinh/model-weights-autoshot/ckpt_0_200_0.pth"

def process_all_batches(parent_folder, model_weights):
    
    # 1. Tìm tất cả các thư mục con (L21_a, L22_a,...) bên trong thư mục gốc
    # Sử dụng sorted để chạy tuần tự từ nhỏ đến lớn
    batch_folders = sorted([
        f for f in glob.glob(os.path.join(parent_folder, "L*")) 
        if os.path.isdir(f)
    ])

    batch_folders = batch_folders[-2:]
    
    if not batch_folders:
        print(f"Cảnh báo: Không tìm thấy thư mục dạng 'L*' nào trong {parent_folder}")
        return
        
    print(f"-> Phát hiện tổng cộng {len(batch_folders)} thư mục batch cần cày cuốc.\n")
    
    # [QUAN TRỌNG TỐI ĐA] Chỉ khởi tạo Model 1 lần duy nhất ngoài vòng lặp để tối ưu VRAM
    print("-> Đang nạp Model AutoShot vào GPU...")
    shot_detector = AutoShot(pretrained_path=model_weights)
    print("-> Nạp Model thành công!\n")
    
    # 2. Vòng lặp quét qua từng thư mục batch
    for b_idx, folder_path in enumerate(batch_folders):
        batch_name = os.path.basename(folder_path)
        print(f"\n==================================================")
        print(f" PROGRESS: Thư mục [{b_idx + 1}/{len(batch_folders)}] -> {batch_name}")
        print(f"==================================================")
        
        # Tự động tạo file JSON kết quả riêng cho từng thư mục (Ví dụ: all_scenes_L21_a.json)
        output_json = f"/kaggle/working/all_scenes_{batch_name}.json"
        
        # Tìm tất cả file .mp4 trong batch hiện tại
        video_files = glob.glob(os.path.join(folder_path, "*.mp4"))
        if not video_files:
            print(f" -> Không có video .mp4 nào trong {batch_name}. Bỏ qua...")
            continue
            
        print(f" -> Tìm thấy {len(video_files)} video. Bắt đầu xử lý...")
        
        all_results = {}
        
        # 3. Vòng lặp cày qua từng video trong batch hiện tại
        for v_idx, vid_path in enumerate(video_files):
            vid_name = os.path.basename(vid_path)
            print(f"   [{v_idx + 1}/{len(video_files)}] {vid_name}")
            
            try:
                # Đọc video & detect
                frames = get_frames(vid_path)
                scenes = shot_detector.detect_shots(frames)
                all_results[vid_name] = scenes
                print(f"      => OK! {len(scenes)} cảnh.")
                
            except Exception as e:
                print(f"      => LỖI: {str(e)}. Bỏ qua...")
                all_results[vid_name] = "ERROR"
                
            finally:
                # Giải phóng bộ nhớ liên tục
                if 'frames' in locals():
                    del frames
                gc.collect()
                torch.cuda.empty_cache()
                
            # Ghi snapshot liên tục phòng trường hợp ngắt quãng
            with open(output_json, "w", encoding="utf-8") as f:
                json.dump(all_results, f, ensure_ascii=False, indent=4)
                
        print(f" -> Hoàn thành {batch_name}! Kết quả lưu tại: {output_json}")


# Kích hoạt hệ thống
process_all_batches(PARENT_VIDEO_DIR, WEIGHTS_PATH)

-> Phát hiện tổng cộng 2 thư mục batch cần cày cuốc.

-> Đang nạp Model AutoShot vào GPU...
-> Nạp Model thành công!


 PROGRESS: Thư mục [1/2] -> L29_a
 -> Tìm thấy 23 video. Bắt đầu xử lý...
   [1/23] L29_V008.mp4


Chạy Model AutoShot: 0batch [00:00, ?batch/s]

      => OK! 209 cảnh.
   [2/23] L29_V003.mp4


Chạy Model AutoShot: 0batch [00:00, ?batch/s]

      => OK! 150 cảnh.
   [3/23] L29_V012.mp4


...

      => OK! 28 cảnh.
   [95/96] L30_V057.mp4


Chạy Model AutoShot: 0batch [00:00, ?batch/s]

      => OK! 20 cảnh.
   [96/96] L30_V053.mp4


Chạy Model AutoShot: 0batch [00:00, ?batch/s]

      => OK! 47 cảnh.
 -> Hoàn thành L30_a! Kết quả lưu tại: /kaggle/working/all_scenes_L30_a.json


In [6]:
# VIDEO_DIR = "/kaggle/input/datasets/dtdat1725/aic-hcmc-2025-batch-1/video/L21_a" 
# WEIGHTS_PATH = "/kaggle/input/datasets/anhnguynnhtinh/model-weights-autoshot/ckpt_0_200_0.pth"

# # File JSON để lưu lại toàn bộ kết quả của hàng trăm video
# OUTPUT_JSON = "/kaggle/working/all_videos_scenes.json"

# def process_multiple_videos(video_folder, model_weights):
#     print("=== KHỞI ĐỘNG CỖ MÁY XỬ LÝ HÀNG LOẠT ===")
    
#     # [QUAN TRỌNG TỐI ĐA] Chỉ khởi tạo Model 1 lần duy nhất đẩy vào GPU
#     print("-> Đang nạp Model AutoShot vào GPU...")
#     shot_detector = AutoShot(pretrained_path=model_weights)
#     print("-> Nạp Model thành công!\n")
    
#     # Tìm tất cả các file .mp4 trong thư mục (có thể thêm .avi, .mkv nếu cần)
#     video_files = glob.glob(os.path.join(video_folder, "*.mp4"))
    
#     if not video_files:
#         print(f"Cảnh báo: Không tìm thấy video .mp4 nào trong {video_folder}")
#         return
        
#     print(f"-> Phát hiện tổng cộng {len(video_files)} video cần xử lý.\n")
    
#     # Dictionary lưu trữ kết quả: { "ten_video_1.mp4": [[0,10], [11,20]], ... }
#     all_results = {}
    
#     # Vòng lặp cày cuốc qua từng video
#     for idx, vid_path in enumerate(video_files):
#         vid_name = os.path.basename(vid_path)
#         print(f"[{idx + 1}/{len(video_files)}] Đang xử lý: {vid_name}")
        
#         try:
#             # 1. Đọc video
#             frames = get_frames(vid_path)
            
#             # 2. Phát hiện phân cảnh
#             scenes = shot_detector.detect_shots(frames)
            
#             # 3. Lưu vào biến tổng
#             all_results[vid_name] = scenes
#             print(f"    => Hoàn thành! Tìm thấy {len(scenes)} cảnh.")
            
#         except Exception as e:
#             # Nếu video bị hỏng (corrupt) hoặc lỗi, báo lỗi và nhảy sang video tiếp theo
#             print(f"    => LỖI ở video {vid_name}: {str(e)}. Đang bỏ qua...")
#             all_results[vid_name] = "ERROR"
            
#         finally:
#             # [QUAN TRỌNG TỐI ĐA] Dọn rác RAM và VRAM sau mỗi video để tránh sập (OOM)
#             if 'frames' in locals():
#                 del frames
#             gc.collect()
#             torch.cuda.empty_cache()
            
#         # [MẸO NHỎ] Lưu File JSON liên tục sau mỗi video. 
#         # Lỡ Kaggle có sập ở video số 99, bạn vẫn có kết quả của 98 video trước đó!
#         with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
#             json.dump(all_results, f, ensure_ascii=False, indent=4)
            
#     print(f"\n=== HOÀN TẤT! Đã xử lý xong {len(video_files)} video ===")
#     print(f"-> Toàn bộ kết quả đã được lưu an toàn tại: {OUTPUT_JSON}")

# # Kích hoạt chạy
# process_multiple_videos(VIDEO_DIR, WEIGHTS_PATH)

In [7]:
# VIDEO_INPUT = "/kaggle/input/datasets/trnhnguynthnh/video-autoshot/L21_V001.mp4" 
# WEIGHTS_PATH = "/kaggle/input/datasets/anhnguynnhtinh/model-weights-autoshot/ckpt_0_200_0.pth"

In [8]:
# print("BƯỚC 1: Đang đọc video")
# frames = get_frames(VIDEO_INPUT) 
# print(f"-> Đã tải {len(frames)} frame.")
    
# print("BƯỚC 2: Khởi chạy Model AutoShot (Inference)")
# shot_detector = AutoShot(pretrained_path=WEIGHTS_PATH)
# scenes_output = shot_detector.detect_shots(frames)
    
# print("=== KẾT QUẢ OUTPUT THÔ (RAW SCENES) ===")
# print(f"Tổng số phân cảnh phát hiện được: {len(scenes_output)}")
# print("Danh sách các cảnh [start_frame, end_frame]:")
# print(scenes_output)
    
# raw_scenes = scenes_output

In [9]:
# def visualize_scenes(video_path, scenes, num_scenes_to_show=5):
#     print(f"Đang trích xuất tuần tự hình ảnh để hiển thị {num_scenes_to_show} phân cảnh đầu tiên...")
    
#     scenes_to_show = scenes[:num_scenes_to_show]
    
#     # 1. Lên danh sách chính xác các khung hình cần lấy (để không phải tua nhảy cóc)
#     target_frames = {}
#     for i, (start, end) in enumerate(scenes_to_show):
#         mid = (start + end) // 2
#         target_frames[start] = {"scene_idx": i, "col_idx": 0, "title": f"Scene {i+1} - Start ({start})"}
#         target_frames[mid]   = {"scene_idx": i, "col_idx": 1, "title": f"Scene {i+1} - Mid ({mid})"}
#         target_frames[end]   = {"scene_idx": i, "col_idx": 2, "title": f"Scene {i+1} - End ({end})"}
        
#     # Khung hình xa nhất cần lấy (để dừng video sớm, tiết kiệm thời gian)
#     max_frame_needed = max(target_frames.keys())
    
#     # Tạo bảng vẽ (grid)
#     fig, axes = plt.subplots(len(scenes_to_show), 3, figsize=(15, 3 * len(scenes_to_show)))
#     if len(scenes_to_show) == 1:
#         axes = [axes]
        
#     # 2. Đọc video tuần tự từ đầu đến max_frame_needed
#     cap = cv2.VideoCapture(video_path)
#     current_frame = 0
    
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret or current_frame > max_frame_needed:
#             break
            
#         # Nếu khung hình hiện tại nằm trong danh sách cần vẽ, vẽ nó luôn!
#         if current_frame in target_frames:
#             info = target_frames[current_frame]
#             i = info["scene_idx"]
#             j = info["col_idx"]
            
#             frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#             axes[i][j].imshow(frame_rgb)
#             axes[i][j].set_title(info["title"], fontsize=12, fontweight='bold')
#             axes[i][j].axis('off')
            
#         current_frame += 1
        
#     cap.release()
#     plt.tight_layout()
#     plt.show()

In [10]:
# visualize_scenes(VIDEO_INPUT, raw_scenes, num_scenes_to_show=5)